# STANCEOSAURUS BENCHMARK: BERTWEET BASELINE (SOTA EMNLP 2022)
Notebook này thực hiện huấn luyện và đánh giá mô hình baseline **BERTweet** (EMNLP 2022) trên tập dữ liệu hội thoại đa bên đã chuẩn hóa **Stanceosaurus Multi-party English** (`train_conversations.json`, `dev_conversations.json`, `test_conversations.json`).

### ⚖️ NGUYÊN TẮC CÔNG BẰNG (FAIR BENCHMARK PROTOCOL):
1. **Cùng Backbone:** Sử dụng `vinai/bertweet-base` (RoBERTa pre-trained trên 850M English tweets).
2. **Cùng Dữ Liệu:** Nạp đúng tập dữ liệu hội thoại đa bên đã làm sạch và cân bằng độ sâu ($K=6$).
3. **Cùng Phân Chia Unseen Claims:** Đánh giá Zero-shot trên 44 claims hoàn toàn mới của tập Test.
4. **Cùng Effective Batch Size:** Batch size 16 với 2 bước gradient accumulation = 32.
5. **Cùng Hàm Mất Mát:** Áp dụng Class-Balanced Focal Loss ($\gamma = 2.0$, trọng số nghịch đảo căn bậc hai tần suất lớp) để xử lý mất cân bằng tự nhiên.

### 💡 CƠ CHẾ CHẠY TEST NHANH VÀ TREO MÁY QUA ĐÊM:
- **Chạy thử siêu tốc (Smoke Test):** Đặt `DEBUG_MODE = True` ở Cell 2. Code sẽ chỉ nạp 40 mẫu/lớp và chạy 1 epoch trong ~30 giây.
- **Treo máy qua đêm:** Đổi thành `DEBUG_MODE = False`, nhấn **Save Version** -> **Save & Run All (Commit)** trên Kaggle.

In [ ]:
# ==============================================================================
# STEP 1: CÀI ĐẶT THƯ VIỆN & TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN DỮ LIỆU
# ==============================================================================
!pip install -q transformers openpyxl emoji python-box scikit-learn

import os
import glob
import json
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm.auto import tqdm
from box import Box

# Tự động dò tìm file dữ liệu train_conversations.json trên Kaggle hoặc Local
found_files = glob.glob('/kaggle/input/**/train_conversations.json', recursive=True) + \
              glob.glob('./**/train_conversations.json', recursive=True) + \
              glob.glob('D:/KLTN/**/train_conversations.json', recursive=True) + \
              glob.glob('d:/Khóa Luận Tốt Nghiệp/**/train_conversations.json', recursive=True)

if not found_files:
    raise FileNotFoundError('❌ Không tìm thấy train_conversations.json! Hãy kiểm tra Input Dataset trên Kaggle.')

DATA_DIR = os.path.dirname(found_files[0])
print(f'--> ✅ Đã kết nối dữ liệu Stanceosaurus Reconstructed tại: {DATA_DIR}')


In [ ]:
# ==============================================================================
# STEP 2: CẤU HÌNH SIÊU THAM SỐ (CHUẨN ĐỐI SOÁT CÔNG BẰNG)
# ==============================================================================
DEBUG_MODE = False  # <-- ĐỔI THÀNH False KHI MUỐN TREO MÁY QUA ĐÊM TRÊN KAGGLE!

config = Box({
    'data_dir': DATA_DIR,
    'debug_mode': DEBUG_MODE,
    'debug_samples_per_class': 40,        # 40 mẫu/lớp khi chạy thử siêu tốc
    'model_name': 'vinai/bertweet-base',  # Backbone BERTweet chuẩn domain Twitter
    'num_classes': 3,                     # 3 lớp: Supporting: 0, Refuting: 1, Other: 2
    'max_seq_len': 128,                   # BERTweet sequence length tối ưu
    'batch_size': 16,                     # Batch size mẫu hội thoại
    'accum_steps': 2,                     # Effective batch size = 32
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'num_epochs': 1 if DEBUG_MODE else 25,  # Trần epoch cao, dừng thật bằng Early Stopping (xem early_stop_patience)
    'early_stop_patience': 4,             # Dừng sớm nếu Dev Macro-F1 không cải thiện sau 4 epoch liên tiếp
    'warmup_ratio': 0.1,
    'loss_type': 'focal',                 # Class-Balanced Focal Loss
    'focal_gamma': 2.0,
    'use_amp': True,                      # Mixed Precision FP16
    'seed': 42,
    'output_dir': './',
    'best_model_path': './best_bertweet_stanceosaurus.pt'
})

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)
config.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

if config.debug_mode:
    print('⚠️ CHẾ ĐỘ: [DEBUG MODE] - Chạy thử nhanh trong 30 giây!')
else:
    print('🔥 CHẾ ĐỘ: [FULL TRAINING] - Chạy toàn bộ dữ liệu!')
print(f'--> 🚀 Thiết bị: {config.device} | Max Epochs: {config.num_epochs} (early stop patience={config.early_stop_patience}) | Model: {config.model_name}')


In [ ]:
# ==============================================================================
# STEP 3: NẠP DỮ LIỆU HỘI THOẠI JSON & CHUYỂN ĐỔI SANG DẠNG SENTENCE-PAIR
# ==============================================================================
def load_conversation_dataframe(filename):
    file_path = os.path.join(config.data_dir, filename)
    with open(file_path, 'r', encoding='utf-8') as f:
        samples = json.load(f)
        
    records = []
    for s in samples:
        claim = s['target']
        sentences = s['sentences']
        target_text = sentences[-1] # Phát ngôn đích cần phân loại lập trường
        context = ' </s> '.join(sentences[:-1]) if len(sentences) > 1 else ''
        
        # Format chuẩn theo bài báo Stanceosaurus: [CLS] claim [SEP] target_text [SEP] context
        if context:
            full_text = f'{target_text} </s> {context}'
        else:
            full_text = target_text
            
        records.append({
            'claim': claim,
            'text': full_text,
            'label': int(s['label'])
        })
    return pd.DataFrame(records)

print('[*] Đang đọc dữ liệu hội thoại JSON...')
train_df = load_conversation_dataframe('train_conversations.json')
dev_df = load_conversation_dataframe('dev_conversations.json')
test_df = load_conversation_dataframe('test_conversations.json')

if config.debug_mode:
    k = config.debug_samples_per_class
    train_df = train_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), k), random_state=config.seed)).reset_index(drop=True)
    dev_df = dev_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)
    test_df = test_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)
    print(f'⚠️ [DEBUG MODE] Đã trích mẫu cân bằng: Train={len(train_df)}, Dev={len(dev_df)}, Test={len(test_df)}')
else:
    print(f'--> [FULL DATA] Train : {len(train_df):,} mẫu | {train_df["claim"].nunique()} claims')
    print(f'--> [FULL DATA] Dev   : {len(dev_df):,} mẫu | {dev_df["claim"].nunique()} claims')
    print(f'--> [FULL DATA] Test  : {len(test_df):,} mẫu | {test_df["claim"].nunique()} claims')

label_names = ['Supporting', 'Refuting', 'Other']
dist_df = pd.DataFrame({
    'Train': train_df['label'].value_counts().sort_index(),
    'Dev': dev_df['label'].value_counts().sort_index(),
    'Test': test_df['label'].value_counts().sort_index()
}, index=range(len(label_names)))
dist_df.index = label_names
display(dist_df)


In [ ]:
# ==============================================================================
# STEP 4: TOKENIZER VÀ PYTORCH DATALOADER
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)

class StanceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.claims = df['claim'].tolist()
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.claims[idx],
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(StanceDataset(train_df, tokenizer, config.max_seq_len), batch_size=config.batch_size, shuffle=True, num_workers=2)
dev_loader = DataLoader(StanceDataset(dev_df, tokenizer, config.max_seq_len), batch_size=config.batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(StanceDataset(test_df, tokenizer, config.max_seq_len), batch_size=config.batch_size, shuffle=False, num_workers=2)
print('--> ✅ DataLoader BERTweet đã khởi tạo thành công!')


In [ ]:
# ==============================================================================
# STEP 5: HÀM MẤT MÁT (CLASS-BALANCED FOCAL LOSS)
# ==============================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weights=None):
        super().__init__()
        self.gamma = gamma
        self.weights = weights

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weights, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Tính trọng số lớp nghịch đảo căn bậc hai (Inverse Square Root) để cân bằng
counts = [sum(train_df['label'] == c) for c in range(config.num_classes)]
class_weights = torch.tensor([1.0 / (max(c, 1) ** 0.5) for c in counts], dtype=torch.float).to(config.device)
class_weights = class_weights / class_weights.sum()

if config.loss_type == 'focal':
    criterion = FocalLoss(gamma=config.focal_gamma, weights=class_weights)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f'--> Trọng số lớp (Class Weights): {class_weights.cpu().numpy().round(3)}')


In [ ]:
# ==============================================================================
# STEP 6: MÔ HÌNH BERTWEET CLASSIFIER & TỐI ƯU HÓA
# ==============================================================================
class BERTweetStanceClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_dim = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]  # Token <s>
        logits = self.classifier(self.dropout(cls_rep))
        return logits

model = BERTweetStanceClassifier(config.model_name, config.num_classes).to(config.device)

optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
total_steps = (len(train_loader) // config.accum_steps) * config.num_epochs
warmup_steps = int(total_steps * config.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=max(total_steps, 1))
scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp)
print(f'--> ✅ Tổng số bước huấn luyện: {total_steps} (Warmup: {warmup_steps})')


In [ ]:
# ==============================================================================
# STEP 7: VÒNG LẶP HUẤN LUYỆN VÀ VALIDATION
# ==============================================================================
def evaluate_model(model, loader, target_names):
    model.eval()
    all_preds, all_labels = [], []
    cls_list = list(range(len(target_names)))
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            labels = batch['label'].to(config.device)

            with torch.cuda.amp.autocast(enabled=config.use_amp):
                logits = model(input_ids, attention_mask)
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, labels=cls_list, average='macro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    per_class_f1 = f1_score(all_labels, all_preds, labels=cls_list, average=None, zero_division=0)
    report = classification_report(all_labels, all_preds, labels=cls_list, target_names=target_names, zero_division=0, output_dict=True)
    return macro_f1, acc, per_class_f1, report

best_dev_f1 = 0.0
patience_counter = 0
history = []
best_model_path = config.best_model_path

print('\n[*] BẮT ĐẦU TRAINING BERTWEET BASELINE...')
for epoch in range(1, config.num_epochs + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    loop = tqdm(train_loader, desc=f'Epoch {epoch}/{config.num_epochs}')
    for step, batch in enumerate(loop):
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        labels = batch['label'].to(config.device)

        with torch.cuda.amp.autocast(enabled=config.use_amp):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels) / config.accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * config.accum_steps

        if (step + 1) % config.accum_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        loop.set_postfix(loss=total_loss / (step + 1))

    avg_train_loss = total_loss / len(train_loader)
    dev_f1, dev_acc, dev_per_f1, _ = evaluate_model(model, dev_loader, label_names)
    
    print(f'--> [Epoch {epoch}] Train Loss: {avg_train_loss:.4f} | Dev Macro-F1: {dev_f1*100:.2f}% | Dev Acc: {dev_acc*100:.2f}%')
    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'dev_macro_f1': dev_f1, 'dev_acc': dev_acc})
    
    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f'    🏆 [Lưu Checkpoint Tốt Nhất] Dev F1 đạt: {best_dev_f1*100:.2f}%')
    else:
        patience_counter += 1
        print(f'    ⏳ [Không cải thiện] {patience_counter}/{config.early_stop_patience} epoch liên tiếp (Best: {best_dev_f1*100:.2f}%)')
        if patience_counter >= config.early_stop_patience:
            print(f'    🛑 [EARLY STOPPING] Dừng tại epoch {epoch} vì Dev Macro-F1 không cải thiện sau {config.early_stop_patience} epoch liên tiếp.')
            break


In [ ]:
# ==============================================================================
# STEP 8: ĐÁNH GIÁ TRÊN TẬP TEST ZERO-SHOT & XUẤT KẾT QUẢ
# ==============================================================================
print('\n' + '='*75)
print('[*] TIẾN HÀNH ĐÁNH GIÁ TRÊN TẬP TEST ZERO-SHOT VỚI CHECKPOINT TỐT NHẤT')
print('='*75)

best_model_path = getattr(config, 'best_model_path', './best_bertweet_stanceosaurus.pt')

if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=config.device))
    print(f'--> [✓] Đã nạp thành công checkpoint tốt nhất từ: {best_model_path}')
else:
    print('--> [!] Không tìm thấy file checkpoint đã lưu, sử dụng trực tiếp trọng số hiện tại của model.')

test_f1, test_acc, test_per_f1, test_report_dict = evaluate_model(model, test_loader, label_names)

res_dict = {
    'Model': 'BERTweet (Baseline EMNLP 2022)',
    'Mode': 'DEBUG' if config.debug_mode else 'FULL',
    'Num_Classes': config.num_classes,
    'Loss_Type': config.loss_type,
    'Accuracy': round(test_acc * 100, 2),
    'Macro_F1': round(test_f1 * 100, 2)
}

for idx, name in enumerate(label_names):
    if idx < len(test_per_f1):
        res_dict[f'F1_{name}'] = round(test_per_f1[idx] * 100, 2)

results_df = pd.DataFrame([res_dict])
results_df.to_csv('stanceosaurus_bertweet_results.csv', index=False)

print('\n--- 📊 BẢNG KẾT QUẢ THỰC NGHIỆM BERTWEET (BASELINE) ---')
display(results_df)

report_df = pd.DataFrame(test_report_dict).T
print('\n--- Báo Cáo Phân Loại Chi Tiết Từng Nhãn ---')
display(report_df.round(3))

if config.debug_mode:
    print('\n🎉 [XÁC NHẬN] CHẾ ĐỘ DEBUG ĐÃ CHẠY THÀNH CÔNG 100% KHÔNG LỖI!')
    print('👉 BẠN CHỈ CẦN QUAY LẠI CELL 2, ĐỔI `DEBUG_MODE = False`, RỒI NHẤN COMMIT TREO QUA ĐÊM!')
else:
    print('\n✅ [HOÀN TẤT] ĐÃ CHẠY XONG FULL DỮ LIỆU QUA ĐÊM!')
